# YOLO Instance Segmentation — AI Innovator Challenge

This notebook converts existing bounding-box annotations into instance-segmentation annotations.

Pipeline:

Bounding-box JSON  
→ SAM 2.1 using bounding boxes as prompts  
→ segmentation masks  
→ polygon annotations  
→ human review/refinement  
→ YOLO segmentation dataset

The original bounding-box JSON files are preserved and are not overwritten.

In [ ]:
from pathlib import Path
import json
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch

from ultralytics import SAM

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    device = 0
    print("GPU:", torch.cuda.get_device_name(0))
else:
    device = "cpu"
    print("Using CPU")

In [ ]:
# =========================
# PROJECT PATHS
# =========================

segmentation_project = Path.cwd()

# Main AIIC folder
project_path = segmentation_project.parent

# Original images + bbox JSON
clean_photos_path = project_path / "clean_photos"

# New segmentation annotations
segmentation_output = (
    segmentation_project
    / "segmentation_annotations"
)

segmentation_output.mkdir(
    parents=True,
    exist_ok=True
)

print("Main AIIC project :", project_path)
print("Segmentation work :", segmentation_project)
print("Source images     :", clean_photos_path)
print("Segmentation output:", segmentation_output)

print("\nImages folder exists:", clean_photos_path.exists())

In [ ]:
# =========================
# LOAD SAM 2.1
# =========================

sam = SAM("sam2.1_b.pt")

print("SAM model loaded successfully.")

In [ ]:
# =========================
# COLLECT ALL REMAINING IMAGES
# =========================

image_exts = {".jpg", ".jpeg", ".png"}

all_pending = []

for class_folder in sorted(clean_photos_path.iterdir()):

    if not class_folder.is_dir():
        continue

    class_name = class_folder.name

    class_segmentation_folder = (
        segmentation_output / class_name
    )

    class_segmentation_folder.mkdir(
        parents=True,
        exist_ok=True
    )

    for img in sorted(class_folder.iterdir()):

        if not img.is_file():
            continue

        if img.suffix.lower() not in image_exts:
            continue

        bbox_json = img.with_suffix(".json")

        segmentation_json = (
            class_segmentation_folder
            / f"{img.stem}.json"
        )

        # Must have bbox annotation
        if not bbox_json.exists():
            continue

        # Skip already segmented
        if segmentation_json.exists():
            continue

        all_pending.append({
            "class_name": class_name,
            "image_path": img,
            "bbox_json": bbox_json,
            "output_json": segmentation_json
        })


print("Total remaining images:", len(all_pending))

print("\nFirst 20 pending:")
for i, item in enumerate(all_pending[:20], start=1):
    print(
        f"{i:02}. "
        f"[{item['class_name']}] "
        f"{item['image_path'].name}"
    )

In [ ]:
# =========================
# HELPER FUNCTIONS
# =========================

def load_bbox_annotation(json_path):
    """
    Load rectangle annotations from X-AnyLabeling JSON.
    Supports both 2-point and 4-corner rectangles.
    """

    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    boxes = []
    labels = []

    for shape in data.get("shapes", []):

        if shape.get("shape_type") != "rectangle":
            continue

        points = np.array(
            shape["points"],
            dtype=float
        )

        x_min = points[:, 0].min()
        y_min = points[:, 1].min()
        x_max = points[:, 0].max()
        y_max = points[:, 1].max()

        boxes.append([
            float(x_min),
            float(y_min),
            float(x_max),
            float(y_max)
        ])

        labels.append(shape["label"])

    return boxes, labels


def mask_to_polygon(mask, image_width, image_height):
    """
    Convert a SAM mask into one simplified polygon.
    """

    if mask.shape != (image_height, image_width):

        mask = cv2.resize(
            mask,
            (image_width, image_height),
            interpolation=cv2.INTER_NEAREST
        )

    mask_uint8 = (
        (mask > 0.5).astype(np.uint8) * 255
    )

    contours, _ = cv2.findContours(
        mask_uint8,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    if not contours:
        return None

    # Take largest contour
    contour = max(
        contours,
        key=cv2.contourArea
    )

    # Ignore tiny accidental masks
    if cv2.contourArea(contour) < 20:
        return None

    # Simplify polygon
    epsilon = (
        0.002
        * cv2.arcLength(contour, True)
    )

    approx = cv2.approxPolyDP(
        contour,
        epsilon,
        True
    )

    polygon = (
        approx
        .reshape(-1, 2)
        .astype(float)
        .tolist()
    )

    if len(polygon) < 3:
        return None

    return polygon

In [ ]:
# =========================
# PROCESS ALL REMAINING IMAGES
# =========================

all_results = []

total = len(all_pending)

for index, item in enumerate(all_pending, start=1):

    class_name = item["class_name"]
    image_path = item["image_path"]
    bbox_json_path = item["bbox_json"]
    output_json = item["output_json"]

    print(
        f"\n[{index}/{total}] "
        f"[{class_name}] "
        f"{image_path.name}"
    )

    # -------------------------
    # Load image
    # -------------------------

    image_bgr = cv2.imread(str(image_path))

    if image_bgr is None:
        print("❌ IMAGE ERROR")

        all_results.append({
            "class": class_name,
            "image": image_path.name,
            "status": "IMAGE ERROR"
        })
        continue

    image_height, image_width = image_bgr.shape[:2]

    # -------------------------
    # Load bbox
    # -------------------------

    boxes, labels = load_bbox_annotation(
        bbox_json_path
    )

    print("BBox objects:", len(boxes))

    if not boxes:
        print("⚠ NO BBOX")

        all_results.append({
            "class": class_name,
            "image": image_path.name,
            "status": "NO BBOX"
        })
        continue

    # -------------------------
    # SAM
    # -------------------------

    try:
        results = sam.predict(
            source=str(image_path),
            bboxes=boxes,
            device=device,
            verbose=False
        )

    except Exception as e:
        print("❌ SAM ERROR:", e)

        all_results.append({
            "class": class_name,
            "image": image_path.name,
            "bbox_count": len(boxes),
            "status": "SAM ERROR"
        })
        continue

    sam_result = results[0]

    if sam_result.masks is None:
        print("⚠ NO MASK")

        all_results.append({
            "class": class_name,
            "image": image_path.name,
            "bbox_count": len(boxes),
            "status": "NO MASK"
        })
        continue

    masks = (
        sam_result.masks.data
        .cpu()
        .numpy()
    )

    print("SAM masks:", len(masks))

    # -------------------------
    # Mask -> Polygon
    # -------------------------

    polygon_shapes = []

    usable_count = min(
        len(masks),
        len(labels)
    )

    for i in range(usable_count):

        polygon = mask_to_polygon(
            masks[i],
            image_width,
            image_height
        )

        if polygon is None:
            print(
                f"⚠ Object {i + 1}: "
                "polygon failed"
            )
            continue

        polygon_shapes.append({
            "label": labels[i],
            "points": polygon,
            "group_id": None,
            "description": "",
            "shape_type": "polygon",
            "flags": {}
        })

    # -------------------------
    # Save JSON
    # -------------------------

    segmentation_data = {
        "version": "5.0.1",
        "flags": {},
        "shapes": polygon_shapes,
        "imagePath": image_path.name,
        "imageData": None,
        "imageHeight": image_height,
        "imageWidth": image_width
    }

    with open(
        output_json,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            segmentation_data,
            f,
            indent=2
        )

    status = (
        "OK"
        if len(boxes) == len(masks) == len(polygon_shapes)
        else "REVIEW"
    )

    print("✅ Saved:", output_json.name)
    print("Polygons:", len(polygon_shapes))
    print("Status:", status)

    all_results.append({
        "class": class_name,
        "image": image_path.name,
        "bbox_count": len(boxes),
        "mask_count": len(masks),
        "polygon_count": len(polygon_shapes),
        "status": status
    })

In [ ]:
# =========================
# GLOBAL SUMMARY
# =========================

from collections import Counter

status_counts = Counter(
    result["status"]
    for result in all_results
)

print("\n" + "=" * 70)
print("GLOBAL SEGMENTATION SUMMARY")
print("=" * 70)

for status, count in status_counts.items():
    print(f"{status}: {count}")

print("\nImages needing attention:")

for result in all_results:
    if result["status"] != "OK":
        print(
            f"[{result['class']}] "
            f"{result['image']} "
            f"-> {result['status']}"
        )

In [ ]:
# =========================
# OVERALL SEGMENTATION PROGRESS
# =========================

total_bbox_ready = 0
total_segmented = 0

for class_folder in clean_photos_path.iterdir():

    if not class_folder.is_dir():
        continue

    class_name = class_folder.name
    class_seg_folder = segmentation_output / class_name

    for img in class_folder.iterdir():

        if not img.is_file():
            continue

        if img.suffix.lower() not in image_exts:
            continue

        if not img.with_suffix(".json").exists():
            continue

        total_bbox_ready += 1

        seg_json = (
            class_seg_folder
            / f"{img.stem}.json"
        )

        if seg_json.exists():
            total_segmented += 1


print("Images with bbox JSON :", total_bbox_ready)
print("Segmentation completed:", total_segmented)
print("Remaining             :", total_bbox_ready - total_segmented)

In [ ]:
# =========================
# FULL BBOX STATUS CHECK
# =========================

total_images = 0
valid_bbox = 0
missing_bbox_json = []
empty_bbox_json = []

for class_folder in sorted(clean_photos_path.iterdir()):

    if not class_folder.is_dir():
        continue

    for img in sorted(class_folder.iterdir()):

        if not img.is_file():
            continue

        if img.suffix.lower() not in image_exts:
            continue

        total_images += 1

        bbox_json = img.with_suffix(".json")

        # CASE 1: JSON langsung tak ada
        if not bbox_json.exists():
            missing_bbox_json.append(img)
            continue

        # CASE 2: JSON ada, check rectangle
        boxes, labels = load_bbox_annotation(bbox_json)

        if len(boxes) == 0:
            empty_bbox_json.append(img)
        else:
            valid_bbox += 1


print("=" * 60)
print("FULL BBOX STATUS")
print("=" * 60)

print("Total images              :", total_images)
print("Valid bbox annotations    :", valid_bbox)
print("Missing bbox JSON         :", len(missing_bbox_json))
print("JSON exists but no bbox   :", len(empty_bbox_json))

print("\nCHECK:")
print(
    valid_bbox
    + len(missing_bbox_json)
    + len(empty_bbox_json)
)

In [ ]:
print("\nMISSING BBOX JSON:\n")

for img in missing_bbox_json:
    print(f"[{img.parent.name}] {img.name}")

In [ ]:
# =========================
# MISSING BBOX BY CLASS
# =========================

from collections import Counter

missing_by_class = Counter(
    img.parent.name
    for img in missing_bbox_json
)

print("=" * 60)
print("MISSING BBOX BY CLASS")
print("=" * 60)

for class_name, count in missing_by_class.most_common():
    print(f"{count:3}  |  {class_name}")

print("\nTotal missing:", sum(missing_by_class.values()))
print("Classes affected:", len(missing_by_class))

In [ ]:
# =========================
# PREPARE PRELIMINARY DATASET
# =========================

import random
import shutil
from collections import defaultdict

dataset_path = segmentation_project / "dataset_preliminary"

for split in ["train", "val", "test"]:
    (dataset_path / "images" / split).mkdir(parents=True, exist_ok=True)
    (dataset_path / "labels" / split).mkdir(parents=True, exist_ok=True)

# 109 class names based on folders
class_names = sorted([
    p.name for p in clean_photos_path.iterdir()
    if p.is_dir()
])

class_to_id = {
    name: i for i, name in enumerate(class_names)
}

print("Total classes:", len(class_names))
print("Dataset:", dataset_path)

In [ ]:
# =========================
# BUILD YOLO SEGMENTATION DATASET
# =========================

random.seed(42)

items_by_class = defaultdict(list)

# Collect only images with segmentation JSON
for class_name in class_names:

    image_folder = clean_photos_path / class_name
    seg_folder = segmentation_output / class_name

    if not seg_folder.exists():
        continue

    for img in image_folder.iterdir():

        if img.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
            continue

        seg_json = seg_folder / f"{img.stem}.json"

        if seg_json.exists():
            items_by_class[class_name].append((img, seg_json))


split_counts = defaultdict(int)

for class_name, items in items_by_class.items():

    random.shuffle(items)

    n = len(items)

    train_end = max(1, int(n * 0.70))
    val_end = train_end + max(1, int(n * 0.15))

    splits = {
        "train": items[:train_end],
        "val": items[train_end:val_end],
        "test": items[val_end:]
    }

    # Ensure test isn't empty
    if len(splits["test"]) == 0 and len(splits["train"]) > 1:
        splits["test"].append(splits["train"].pop())

    for split_name, split_items in splits.items():

        for image_path, json_path in split_items:

            with open(json_path, "r", encoding="utf-8") as f:
                data = json.load(f)

            width = data["imageWidth"]
            height = data["imageHeight"]

            yolo_lines = []

            for shape in data.get("shapes", []):

                if shape.get("shape_type") != "polygon":
                    continue

                points = shape["points"]

                if len(points) < 3:
                    continue

                class_id = class_to_id[class_name]

                normalized = []

                for x, y in points:
                    normalized.append(x / width)
                    normalized.append(y / height)

                line = (
                    str(class_id)
                    + " "
                    + " ".join(f"{v:.6f}" for v in normalized)
                )

                yolo_lines.append(line)

            if not yolo_lines:
                continue

            # Avoid duplicate filenames across classes
            output_stem = f"{class_name}__{image_path.stem}"

            image_output = (
                dataset_path / "images" / split_name
                / f"{output_stem}{image_path.suffix.lower()}"
            )

            label_output = (
                dataset_path / "labels" / split_name
                / f"{output_stem}.txt"
            )

            shutil.copy2(image_path, image_output)

            with open(label_output, "w", encoding="utf-8") as f:
                f.write("\n".join(yolo_lines))

            split_counts[split_name] += 1


print("Train:", split_counts["train"])
print("Val  :", split_counts["val"])
print("Test :", split_counts["test"])
print("Total:", sum(split_counts.values()))

In [ ]:
# =========================
# CREATE DATA.YAML
# =========================

import yaml

yaml_data = {
    "path": str(dataset_path),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {
        i: name
        for i, name in enumerate(class_names)
    }
}

yaml_path = dataset_path / "data.yaml"

with open(yaml_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(
        yaml_data,
        f,
        sort_keys=False,
        allow_unicode=True
    )

print("data.yaml:", yaml_path)
print("Classes:", len(class_names))

In [ ]:
# =========================
# TRAIN PRELIMINARY YOLO SEGMENTATION
# =========================

from ultralytics import YOLO

model = YOLO("yolo11n-seg.pt")

train_results = model.train(
    data=str(yaml_path),
    epochs=50,
    imgsz=640,
    batch=8,
    device=0,
    project=str(segmentation_project / "runs"),
    name="preliminary_segmentation",
    patience=10,
    workers=4
)

In [ ]:
# =========================
# LOAD PRELIMINARY MODEL
# =========================

from ultralytics import YOLO

best_model_path = (
    segmentation_project
    / "runs"
    / "preliminary_segmentation"
    / "weights"
    / "best.pt"
)

prelim_model = YOLO(str(best_model_path))

print("Loaded:")
print(best_model_path)


# =========================
# COLLECT IMAGES WITHOUT BBOX JSON
# =========================

missing_images = []

for class_folder in sorted(clean_photos_path.iterdir()):

    if not class_folder.is_dir():
        continue

    for img in sorted(class_folder.iterdir()):

        if not img.is_file():
            continue

        if img.suffix.lower() not in image_exts:
            continue

        bbox_json = img.with_suffix(".json")

        if not bbox_json.exists():
            missing_images.append(img)


print("\nMissing bbox images:", len(missing_images))

In [ ]:
# =========================
# AUTO-SEGMENT MISSING IMAGES
# =========================

auto_results = []

confidence_threshold = 0.25

for index, image_path in enumerate(missing_images, start=1):

    class_name = image_path.parent.name
    expected_class_id = class_to_id[class_name]

    output_folder = segmentation_output / class_name
    output_folder.mkdir(parents=True, exist_ok=True)

    output_json = output_folder / f"{image_path.stem}.json"

    print(
        f"\n[{index}/{len(missing_images)}] "
        f"[{class_name}] {image_path.name}"
    )

    image_bgr = cv2.imread(str(image_path))

    if image_bgr is None:
        print("❌ IMAGE ERROR")

        auto_results.append({
            "class": class_name,
            "image": image_path.name,
            "status": "IMAGE ERROR"
        })

        continue

    image_height, image_width = image_bgr.shape[:2]

    # YOLO segmentation prediction
    results = prelim_model.predict(
        source=str(image_path),
        conf=confidence_threshold,
        device=0,
        verbose=False
    )

    result = results[0]

    polygon_shapes = []
    confidences = []

    if (
        result.masks is not None
        and result.boxes is not None
    ):

        predicted_classes = (
            result.boxes.cls
            .cpu()
            .numpy()
            .astype(int)
        )

        predicted_conf = (
            result.boxes.conf
            .cpu()
            .numpy()
        )

        polygons = result.masks.xy

        for pred_class, conf, polygon in zip(
            predicted_classes,
            predicted_conf,
            polygons
        ):

            # Keep only expected class
            if pred_class != expected_class_id:
                continue

            if polygon is None or len(polygon) < 3:
                continue

            polygon_shapes.append({
                "label": class_name,
                "points": polygon.astype(float).tolist(),
                "group_id": None,
                "description": "",
                "shape_type": "polygon",
                "flags": {}
            })

            confidences.append(float(conf))

    # Save only when at least one valid mask exists
    if len(polygon_shapes) > 0:

        segmentation_data = {
            "version": "5.0.1",
            "flags": {},
            "shapes": polygon_shapes,
            "imagePath": image_path.name,
            "imageData": None,
            "imageHeight": image_height,
            "imageWidth": image_width
        }

        with open(
            output_json,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                segmentation_data,
                f,
                indent=2
            )

        min_conf = min(confidences)

        status = (
            "OK"
            if min_conf >= 0.50
            else "REVIEW"
        )

        print("Masks:", len(polygon_shapes))
        print("Lowest confidence:", round(min_conf, 3))
        print("Status:", status)

    else:

        status = "NO PREDICTION"

        print("⚠ No expected-class mask found.")

    auto_results.append({
        "class": class_name,
        "image": image_path.name,
        "mask_count": len(polygon_shapes),
        "min_conf": (
            min(confidences)
            if confidences
            else None
        ),
        "status": status
    })

In [ ]:
# =========================
# AUTO-ANNOTATION SUMMARY
# =========================

from collections import Counter

status_counts = Counter(
    result["status"]
    for result in auto_results
)

print("\n" + "=" * 70)
print("229 AUTO-SEGMENTATION SUMMARY")
print("=" * 70)

for status, count in status_counts.items():
    print(f"{status}: {count}")


print("\nImages needing review:")

for result in auto_results:

    if result["status"] != "OK":

        print(
            f"[{result['class']}] "
            f"{result['image']} "
            f"-> {result['status']}"
        )

In [ ]:
# =========================
# DIAGNOSE NO-PREDICTION IMAGES
# =========================

no_prediction_samples = [
    r for r in auto_results
    if r["status"] == "NO PREDICTION"
][:10]

for i, item in enumerate(no_prediction_samples, 1):

    image_path = (
        clean_photos_path
        / item["class"]
        / item["image"]
    )

    results = prelim_model.predict(
        source=str(image_path),
        conf=0.10,       # lower threshold for diagnosis
        device=0,
        verbose=False
    )

    result = results[0]

    print(f"\n{i}. Expected: {item['class']}")

    if result.boxes is None or len(result.boxes) == 0:
        print("   ❌ Model detected NOTHING")
        continue

    classes = result.boxes.cls.cpu().numpy().astype(int)
    confs = result.boxes.conf.cpu().numpy()

    for cls_id, conf in zip(classes, confs):
        print(
            f"   → {class_names[cls_id]} "
            f"({conf:.3f})"
        )

In [ ]:
# =========================
# LOAD YOLO-WORLD
# =========================

from ultralytics import YOLOWorld
import re

world_model = YOLOWorld("yolov8s-worldv2.pt")

def folder_to_prompt(class_name):
    # E001_Arduino Uno -> Arduino Uno
    prompt = re.sub(r"^[A-Z]\d{3}_", "", class_name)
    prompt = prompt.replace("_", " ")
    prompt = prompt.replace("×", "x")
    return prompt.strip()# =========================
# LOAD YOLO-WORLD ON CPU
# =========================

from ultralytics import YOLOWorld
import re

world_model = YOLOWorld("yolov8s-worldv2.pt")

# Force model to CPU
world_model.to("cpu")

def folder_to_prompt(class_name):
    # Example:
    # E001_Arduino Uno -> Arduino Uno

    prompt = re.sub(
        r"^[A-Z]\d{3}_",
        "",
        class_name
    )

    prompt = prompt.replace("_", " ")
    prompt = prompt.replace("×", "x")

    return prompt.strip()

print("YOLO-World loaded on CPU.")

print("YOLO-World loaded.")

In [ ]:
# REBUILD MISSING IMAGES

from pathlib import Path

segmentation_project = Path.cwd()
project_path = segmentation_project.parent
clean_photos_path = project_path / "clean_photos"

image_exts = {".jpg", ".jpeg", ".png"}

missing_images = []

for class_folder in sorted(clean_photos_path.iterdir()):
    if not class_folder.is_dir():
        continue

    for img in sorted(class_folder.iterdir()):
        if (
            img.is_file()
            and img.suffix.lower() in image_exts
            and not img.with_suffix(".json").exists()
        ):
            missing_images.append(img)

print("Missing images:", len(missing_images))

In [ ]:
# =========================
# TEST YOLO-WORLD ON 5 IMAGES
# =========================

world_test_images = missing_images[:5]

world_test_results = []

for i, image_path in enumerate(
    world_test_images,
    start=1
):

    class_name = image_path.parent.name
    prompt = folder_to_prompt(class_name)

    print(f"\n{i}. {image_path.name}")
    print("Expected:", class_name)
    print("Prompt  :", prompt)

    # Tell YOLO-World what object to search for
    world_model.set_classes([prompt])

    results = world_model.predict(
        source=str(image_path),
        conf=0.10,
        imgsz=640,
        device="cpu",
        verbose=False
    )

    result = results[0]

    boxes = []

    if result.boxes is not None:

        xyxy = (
            result.boxes.xyxy
            .cpu()
            .numpy()
        )

        confs = (
            result.boxes.conf
            .cpu()
            .numpy()
        )

        for box, conf in zip(
            xyxy,
            confs
        ):

            boxes.append({
                "box": box.tolist(),
                "confidence": float(conf)
            })

    print("Detected objects:", len(boxes))

    for j, item in enumerate(
        boxes,
        start=1
    ):

        print(
            f"   Object {j}: "
            f"confidence = "
            f"{item['confidence']:.3f}"
        )

    world_test_results.append({
        "image_path": image_path,
        "class_name": class_name,
        "prompt": prompt,
        "boxes": boxes
    })

In [ ]:
%pip install -U transformers accelerate safetensors

In [ ]:
# =========================
# LOAD GROUNDING DINO
# =========================

from pathlib import Path
from PIL import Image
import torch
import re

from transformers import (
    AutoProcessor,
    AutoModelForZeroShotObjectDetection
)

segmentation_project = Path.cwd()
project_path = segmentation_project.parent
clean_photos_path = project_path / "clean_photos"

image_exts = {".jpg", ".jpeg", ".png"}

# Rebuild 229 missing images
missing_images = []

for class_folder in sorted(clean_photos_path.iterdir()):

    if not class_folder.is_dir():
        continue

    for img in sorted(class_folder.iterdir()):

        if (
            img.is_file()
            and img.suffix.lower() in image_exts
            and not img.with_suffix(".json").exists()
        ):
            missing_images.append(img)


device = "cuda" if torch.cuda.is_available() else "cpu"

model_id = "IDEA-Research/grounding-dino-tiny"

processor = AutoProcessor.from_pretrained(model_id)

dino_model = (
    AutoModelForZeroShotObjectDetection
    .from_pretrained(model_id)
    .to(device)
)

print("Device:", device)
print("Missing images:", len(missing_images))
print("Grounding DINO loaded.")

In [ ]:
# =========================
# TEST GROUNDING DINO - 5 IMAGES
# =========================

prompt_map = {
    "E001_Arduino Uno": "arduino uno development board",
    "E003_Motor Shield": "arduino motor shield circuit board",
}

def get_prompt(class_name):

    if class_name in prompt_map:
        return prompt_map[class_name]

    prompt = re.sub(
        r"^[A-Z]\d{3}_",
        "",
        class_name
    )

    prompt = prompt.replace("_", " ")
    prompt = prompt.replace("×", "x")

    return prompt.lower().strip()


dino_test_results = []

for i, image_path in enumerate(missing_images[:5], start=1):

    class_name = image_path.parent.name
    prompt = get_prompt(class_name)

    image = Image.open(image_path).convert("RGB")

    print(f"\n{i}. {image_path.name}")
    print("Expected:", class_name)
    print("Prompt  :", prompt)

    text_labels = [[prompt]]

    inputs = processor(
        images=image,
        text=text_labels,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = dino_model(**inputs)

    results = processor.post_process_grounded_object_detection(
        outputs,
        inputs.input_ids,
        threshold=0.20,
        text_threshold=0.15,
        target_sizes=[image.size[::-1]]
    )

    result = results[0]

    boxes = result["boxes"].cpu()
    scores = result["scores"].cpu()

    print("Detected objects:", len(boxes))

    for j, (box, score) in enumerate(
        zip(boxes, scores),
        start=1
    ):
        print(
            f"   Object {j}: "
            f"confidence = {score.item():.3f} "
            f"| box = {[round(x, 1) for x in box.tolist()]}"
        )

    dino_test_results.append({
        "image_path": image_path,
        "class_name": class_name,
        "prompt": prompt,
        "boxes": boxes,
        "scores": scores
    })

In [ ]:
# =========================
# SHOW BEST BOX ONLY (TOP-1)
# =========================

import matplotlib.pyplot as plt
import matplotlib.patches as patches

best_box_results = []

for i, item in enumerate(dino_test_results, start=1):

    image_path = item["image_path"]
    class_name = item["class_name"]
    prompt = item["prompt"]
    boxes = item["boxes"]
    scores = item["scores"]

    if len(boxes) == 0:
        print(f"\n{i}. {image_path.name}")
        print("Expected:", class_name)
        print("Status  : NO DETECTION")
        best_box_results.append({
            "image_path": image_path,
            "class_name": class_name,
            "prompt": prompt,
            "best_box": None,
            "best_score": None
        })
        continue

    best_idx = scores.argmax().item()
    best_box = boxes[best_idx].tolist()
    best_score = scores[best_idx].item()

    best_box_results.append({
        "image_path": image_path,
        "class_name": class_name,
        "prompt": prompt,
        "best_box": best_box,
        "best_score": best_score
    })

    image = Image.open(image_path).convert("RGB")

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(image)

    x1, y1, x2, y2 = best_box
    rect = patches.Rectangle(
        (x1, y1),
        x2 - x1,
        y2 - y1,
        linewidth=2,
        edgecolor="red",
        facecolor="none"
    )
    ax.add_patch(rect)

    ax.text(
        x1,
        max(y1 - 10, 10),
        f"{class_name} | {best_score:.3f}",
        fontsize=10,
        bbox=dict(facecolor="yellow", alpha=0.7)
    )

    ax.set_title(f"{i}. Best Box Only")
    ax.axis("off")
    plt.show()

In [ ]:
# =========================================================
# TEST YOLO-SEG ON 5 MISSING-BBOX IMAGES
# IGNORE CLASS PREDICTION, SHOW ALL MASKS
# =========================================================

from pathlib import Path
from ultralytics import YOLO
import cv2
import numpy as np
import matplotlib.pyplot as plt

project_root = Path(r"G:\AIIC")
clean_photos_path = project_root / "clean_photos"

seg_model_path = (
    project_root
    / "yolo_segmentation"
    / "runs"
    / "preliminary_segmentation"
    / "weights"
    / "best.pt"
)

image_exts = {".jpg", ".jpeg", ".png"}

# Find missing-bbox images
missing_images = []

for class_folder in sorted(clean_photos_path.iterdir()):
    if not class_folder.is_dir():
        continue

    for image_path in sorted(class_folder.iterdir()):
        if (
            image_path.is_file()
            and image_path.suffix.lower() in image_exts
            and not image_path.with_suffix(".json").exists()
        ):
            missing_images.append(
                (class_folder.name, image_path)
            )

print("Missing images:", len(missing_images))

test_images = missing_images[:5]

# Load model
model = YOLO(str(seg_model_path))

conf_threshold = 0.20
iou_threshold = 0.50
min_mask_area = 500

for idx, (class_name, image_path) in enumerate(
    test_images,
    start=1
):

    print("\n" + "=" * 70)
    print(f"{idx}. {image_path.name}")
    print("Expected class:", class_name)

    image_bgr = cv2.imread(str(image_path))
    image_rgb = cv2.cvtColor(
        image_bgr,
        cv2.COLOR_BGR2RGB
    )

    results = model.predict(
        source=str(image_path),
        conf=conf_threshold,
        iou=iou_threshold,
        retina_masks=True,
        device=0,
        verbose=False
    )

    result = results[0]

    if result.masks is None:
        print("❌ No masks detected")

        plt.figure(figsize=(8, 8))
        plt.imshow(image_rgb)
        plt.title(
            f"{class_name}\nNO MASK DETECTED"
        )
        plt.axis("off")
        plt.show()
        continue

    masks = result.masks.data.cpu().numpy()

    # Work in float first for blending
    preview = image_rgb.astype(np.float32)

    valid_masks = []

    for mask in masks:

        bin_mask = (
            mask > 0.5
        ).astype(np.uint8)

        area = int(bin_mask.sum())

        if area < min_mask_area:
            continue

        valid_masks.append(bin_mask)

    cmap = plt.get_cmap("tab20")

    # Overlay masks
    for j, bin_mask in enumerate(valid_masks):

        color = np.array(
            cmap(j % 20)[:3]
        ) * 255

        preview[bin_mask == 1] = (
            0.55 * preview[bin_mask == 1]
            + 0.45 * color
        )

    # IMPORTANT FIX:
    # convert to uint8 BEFORE OpenCV drawing
    preview = np.clip(
        preview,
        0,
        255
    ).astype(np.uint8)

    # Draw contours + numbers
    for j, bin_mask in enumerate(
        valid_masks,
        start=1
    ):

        contours, _ = cv2.findContours(
            bin_mask,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE
        )

        cv2.drawContours(
            preview,
            contours,
            -1,
            (255, 255, 255),
            2
        )

        ys, xs = np.where(
            bin_mask == 1
        )

        if len(xs) > 0:

            cx = int(xs.mean())
            cy = int(ys.mean())

            cv2.putText(
                preview,
                str(j),
                (cx, cy),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 0, 0),
                3,
                cv2.LINE_AA
            )

            cv2.putText(
                preview,
                str(j),
                (cx, cy),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (255, 255, 255),
                1,
                cv2.LINE_AA
            )

    print("Raw masks found   :", len(masks))
    print("Valid masks shown :", len(valid_masks))

    plt.figure(figsize=(9, 9))
    plt.imshow(preview)
    plt.title(
        f"{idx}. {class_name}\n"
        "All masks - predicted class ignored"
    )
    plt.axis("off")
    plt.show()

In [ ]:
# =========================
# AUTO-PREDICTION REVIEW SETUP
# =========================

from pathlib import Path
from ultralytics import YOLO
import cv2
import json
import numpy as np

project_root = Path(r"G:\AIIC")

clean_photos_path = project_root / "clean_photos"

review_output = (
    project_root
    / "yolo_segmentation"
    / "auto_predictions_review"
)

review_output.mkdir(parents=True, exist_ok=True)

model_path = (
    project_root
    / "yolo_segmentation"
    / "runs"
    / "preliminary_segmentation"
    / "weights"
    / "best.pt"
)

model = YOLO(str(model_path))

image_exts = {".jpg", ".jpeg", ".png"}

missing_images = []

for class_folder in sorted(clean_photos_path.iterdir()):

    if not class_folder.is_dir():
        continue

    for img in sorted(class_folder.iterdir()):

        if (
            img.is_file()
            and img.suffix.lower() in image_exts
            and not img.with_suffix(".json").exists()
        ):
            missing_images.append(img)

print("Missing images:", len(missing_images))
print("Review output:", review_output)

In [ ]:
# =========================
# AUTO-PREDICT ALL 229
# =========================

auto_review_results = []

for index, image_path in enumerate(missing_images, start=1):

    class_name = image_path.parent.name

    print(
        f"\n[{index}/{len(missing_images)}] "
        f"[{class_name}] {image_path.name}"
    )

    image_bgr = cv2.imread(str(image_path))

    if image_bgr is None:
        print("❌ IMAGE ERROR")

        auto_review_results.append({
            "class": class_name,
            "image": image_path.name,
            "status": "IMAGE ERROR"
        })
        continue

    h, w = image_bgr.shape[:2]

    results = model.predict(
        source=str(image_path),
        conf=0.20,
        iou=0.50,
        retina_masks=True,
        device=0,
        verbose=False
    )

    result = results[0]

    if result.masks is None or result.boxes is None:
        print("❌ NO MASK")

        auto_review_results.append({
            "class": class_name,
            "image": image_path.name,
            "status": "NO MASK"
        })
        continue

    polygons = result.masks.xy
    confs = result.boxes.conf.cpu().numpy()
    pred_classes = result.boxes.cls.cpu().numpy().astype(int)

    shapes = []

    for polygon, conf, pred_cls in zip(
        polygons,
        confs,
        pred_classes
    ):

        if polygon is None or len(polygon) < 3:
            continue

        # Ignore extremely tiny predictions
        area = cv2.contourArea(
            polygon.astype(np.float32)
        )

        if area < (w * h * 0.0005):
            continue

        shapes.append({
            # True class comes from folder
            "label": class_name,

            "points": polygon.astype(float).tolist(),

            "group_id": None,

            "description": (
                f"AUTO | confidence={float(conf):.3f} | "
                f"predicted_class_id={int(pred_cls)}"
            ),

            "shape_type": "polygon",
            "flags": {}
        })

    if len(shapes) == 0:
        print("❌ NO VALID MASK")

        auto_review_results.append({
            "class": class_name,
            "image": image_path.name,
            "status": "NO MASK"
        })

        continue

    class_output = review_output / class_name
    class_output.mkdir(parents=True, exist_ok=True)

    output_json = (
        class_output
        / f"{image_path.stem}.json"
    )

    data = {
        "version": "5.0.1",
        "flags": {},
        "shapes": shapes,
        "imagePath": image_path.name,
        "imageData": None,
        "imageHeight": h,
        "imageWidth": w
    }

    with open(
        output_json,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(data, f, indent=2)

    print("✅ Generated masks:", len(shapes))

    auto_review_results.append({
        "class": class_name,
        "image": image_path.name,
        "mask_count": len(shapes),
        "status": "GENERATED"
    })

In [ ]:
# =========================
# AUTO-PREDICTION SUMMARY
# =========================

from collections import Counter

counts = Counter(
    x["status"]
    for x in auto_review_results
)

print("\n" + "=" * 60)
print("AUTO-PREDICTION SUMMARY")
print("=" * 60)

for status, count in counts.items():
    print(f"{status}: {count}")

print("\nNO MASK images:")

for item in auto_review_results:
    if item["status"] == "NO MASK":
        print(
            f"[{item['class']}] "
            f"{item['image']}"
        )

In [ ]:
# =========================
# RETRY 75 NO-MASK IMAGES
# LOWER CONFIDENCE
# =========================

lowconf_output = (
    project_root
    / "yolo_segmentation"
    / "auto_predictions_review_lowconf"
)

lowconf_output.mkdir(parents=True, exist_ok=True)

# Get only previous NO MASK images
no_mask_items = [
    item for item in auto_review_results
    if item["status"] == "NO MASK"
]

retry_results = []

print("Images to retry:", len(no_mask_items))

for index, item in enumerate(no_mask_items, start=1):

    class_name = item["class"]

    image_path = (
        clean_photos_path
        / class_name
        / item["image"]
    )

    print(
        f"\n[{index}/{len(no_mask_items)}] "
        f"[{class_name}] {image_path.name}"
    )

    image_bgr = cv2.imread(str(image_path))

    if image_bgr is None:
        retry_results.append({
            "class": class_name,
            "image": image_path.name,
            "status": "IMAGE ERROR"
        })
        continue

    h, w = image_bgr.shape[:2]

    results = model.predict(
        source=str(image_path),
        conf=0.05,       # lower confidence
        iou=0.50,
        retina_masks=True,
        device=0,
        verbose=False
    )

    result = results[0]

    if result.masks is None or result.boxes is None:
        print("❌ STILL NO MASK")

        retry_results.append({
            "class": class_name,
            "image": image_path.name,
            "status": "NO MASK"
        })
        continue

    polygons = result.masks.xy
    confs = result.boxes.conf.cpu().numpy()
    pred_classes = result.boxes.cls.cpu().numpy().astype(int)

    shapes = []

    for polygon, conf, pred_cls in zip(
        polygons,
        confs,
        pred_classes
    ):

        if polygon is None or len(polygon) < 3:
            continue

        area = cv2.contourArea(
            polygon.astype(np.float32)
        )

        if area < (w * h * 0.0005):
            continue

        shapes.append({
            "label": class_name,
            "points": polygon.astype(float).tolist(),
            "group_id": None,
            "description": (
                f"AUTO LOWCONF | confidence={float(conf):.3f} | "
                f"predicted_class_id={int(pred_cls)}"
            ),
            "shape_type": "polygon",
            "flags": {}
        })

    if len(shapes) == 0:
        print("❌ STILL NO VALID MASK")

        retry_results.append({
            "class": class_name,
            "image": image_path.name,
            "status": "NO MASK"
        })
        continue

    class_output = lowconf_output / class_name
    class_output.mkdir(parents=True, exist_ok=True)

    output_json = class_output / f"{image_path.stem}.json"

    data = {
        "version": "5.0.1",
        "flags": {},
        "shapes": shapes,
        "imagePath": image_path.name,
        "imageData": None,
        "imageHeight": h,
        "imageWidth": w
    }

    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)

    print("✅ Generated masks:", len(shapes))

    retry_results.append({
        "class": class_name,
        "image": image_path.name,
        "mask_count": len(shapes),
        "status": "GENERATED"
    })

In [ ]:
from collections import Counter

retry_counts = Counter(
    item["status"]
    for item in retry_results
)

print("\n" + "=" * 60)
print("LOW-CONFIDENCE RETRY SUMMARY")
print("=" * 60)

for status, count in retry_counts.items():
    print(f"{status}: {count}")

In [ ]:
# =========================
# REMAINING NO-MASK BY CLASS
# =========================

from collections import Counter

remaining_no_mask = [
    item for item in retry_results
    if item["status"] == "NO MASK"
]

remaining_by_class = Counter(
    item["class"]
    for item in remaining_no_mask
)

print("=" * 60)
print("REMAINING NO-MASK BY CLASS")
print("=" * 60)

for class_name, count in remaining_by_class.most_common():
    print(f"{count:2} | {class_name}")

print("\nTotal remaining:", len(remaining_no_mask))

In [ ]:
# =========================
# EXACT REMAINING IMAGES
# =========================

for i, item in enumerate(remaining_no_mask, start=1):
    print(
        f"{i:02}. "
        f"[{item['class']}] "
        f"{item['image']}"
    )

In [ ]:
# =========================
# MISSING BBOX BY CLASS
# SORTED BY CLASS NAME
# =========================

from collections import Counter

missing_bbox_images = []

for class_folder in sorted(clean_photos_path.iterdir()):

    if not class_folder.is_dir():
        continue

    for img in sorted(class_folder.iterdir()):

        if (
            img.is_file()
            and img.suffix.lower() in image_exts
            and not img.with_suffix(".json").exists()
        ):
            missing_bbox_images.append(img)


missing_by_class = Counter(
    img.parent.name
    for img in missing_bbox_images
)

print("=" * 60)
print("MISSING BBOX BY CLASS")
print("=" * 60)

for class_name in sorted(missing_by_class):
    print(f"{missing_by_class[class_name]:3} | {class_name}")

print("\nTotal missing images:", len(missing_bbox_images))
print("Classes affected    :", len(missing_by_class))

In [ ]:
# ============================================================
# SAVE INTENTIONAL BBOX EXCLUSIONS
# ============================================================

import json
from pathlib import Path

# We already confirmed:
# total missing bbox = 4
# all 4 are intentional exclusions

exclusions_file = segmentation_project / "intentional_bbox_exclusions.json"

intentional_exclusions = []

for img in sorted(missing_bbox_images):
    intentional_exclusions.append({
        "class": img.parent.name,
        "image": img.name,
        "relative_path": str(img.relative_to(clean_photos_path))
    })

with open(exclusions_file, "w", encoding="utf-8") as f:
    json.dump(intentional_exclusions, f, indent=2, ensure_ascii=False)

print("=" * 60)
print("INTENTIONAL BBOX EXCLUSIONS")
print("=" * 60)

for i, item in enumerate(intentional_exclusions, start=1):
    print(f"{i}. [{item['class']}] {item['image']}")

print("\nTotal exclusions:", len(intentional_exclusions))
print("Saved to:")
print(exclusions_file)

In [ ]:
# ============================================================
# COUNT IMAGES READY FOR SAM SEGMENTATION
# ============================================================

image_exts = {".jpg", ".jpeg", ".png"}

all_images = [
    p for p in clean_photos_path.rglob("*")
    if p.is_file() and p.suffix.lower() in image_exts
]

ready_for_sam = [
    img for img in all_images
    if img.with_suffix(".json").exists()
]

print("=" * 60)
print("SAM SEGMENTATION INPUT")
print("=" * 60)

print("Total images       :", len(all_images))
print("Intentional exclude:", len(intentional_exclusions))
print("Ready for SAM      :", len(ready_for_sam))

In [ ]:
# ============================================================
# FIND IMAGES THAT HAVE BBOX BUT STILL NEED SEGMENTATION
# ============================================================

from pathlib import Path
from collections import Counter

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

project_path = Path(r"G:\AIIC")
clean_photos_path = project_path / "clean_photos"

segmentation_project = project_path / "yolo_segmentation"
segmentation_output = (
    segmentation_project
    / "segmentation_annotations"
)

segmentation_output.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# FIND IMAGES
# ------------------------------------------------------------

image_exts = {
    ".jpg",
    ".jpeg",
    ".png"
}

need_segmentation = []

for image_path in clean_photos_path.rglob("*"):

    # Ignore folders / JSON files
    if not image_path.is_file():
        continue

    if image_path.suffix.lower() not in image_exts:
        continue

    # -------------------------
    # CHECK BBOX JSON
    # -------------------------

    bbox_json = image_path.with_suffix(".json")

    # No bbox = intentional missing / not ready
    if not bbox_json.exists():
        continue

    # -------------------------
    # CHECK SEGMENTATION JSON
    # -------------------------

    class_name = image_path.parent.name

    seg_json = (
        segmentation_output
        / class_name
        / f"{image_path.stem}.json"
    )

    # BBox exists but segmentation does not
    if not seg_json.exists():
        need_segmentation.append(image_path)

# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

need_segmentation = sorted(
    need_segmentation,
    key=lambda p: (
        p.parent.name.lower(),
        p.name.lower()
    )
)

missing_seg_by_class = Counter(
    img.parent.name
    for img in need_segmentation
)

print("=" * 65)
print("IMAGES WITH BBOX BUT WITHOUT SEGMENTATION")
print("=" * 65)

print(
    f"\nTotal images still needing segmentation: "
    f"{len(need_segmentation)}"
)

print(
    f"Classes affected: "
    f"{len(missing_seg_by_class)}"
)

print("\n" + "=" * 65)
print("BY CLASS")
print("=" * 65)

for class_name in sorted(
    missing_seg_by_class,
    key=str.lower
):
    print(
        f"{missing_seg_by_class[class_name]:3} | "
        f"{class_name}"
    )

print("\n" + "=" * 65)
print("FIRST 30 IMAGES")
print("=" * 65)

for i, img in enumerate(
    need_segmentation[:30],
    start=1
):
    print(
        f"{i:02}. "
        f"[{img.parent.name}] "
        f"{img.name}"
    )

if len(need_segmentation) > 30:
    print(
        f"\n... and "
        f"{len(need_segmentation) - 30} more"
    )

In [ ]:
# ============================================================
# CHECK HOW MANY OF THE NEWLY-ANNOTATED GROUP
# ALREADY HAVE SEGMENTATION
# ============================================================

from pathlib import Path
from collections import Counter

project_path = Path(r"G:\AIIC")
clean_photos_path = project_path / "clean_photos"

segmentation_output = (
    project_path
    / "yolo_segmentation"
    / "segmentation_annotations"
)

image_exts = {".jpg", ".jpeg", ".png"}

# All images that currently have bbox
bbox_ready_images = []

for img in clean_photos_path.rglob("*"):
    if (
        img.is_file()
        and img.suffix.lower() in image_exts
        and img.with_suffix(".json").exists()
    ):
        bbox_ready_images.append(img)

# Images that already have segmentation
already_segmented = []

# Images still needing segmentation
still_need_segmentation = []

for img in bbox_ready_images:

    seg_json = (
        segmentation_output
        / img.parent.name
        / f"{img.stem}.json"
    )

    if seg_json.exists():
        already_segmented.append(img)
    else:
        still_need_segmentation.append(img)

print("=" * 65)
print("CURRENT DATASET STATUS")
print("=" * 65)

print("Images with bbox             :", len(bbox_ready_images))
print("Already have segmentation   :", len(already_segmented))
print("Still need segmentation     :", len(still_need_segmentation))

print("\nExpected:")
print("2730 total - 4 intentional skip = 2726 bbox-ready")

print("\nCheck:")
print(
    len(already_segmented),
    "+",
    len(still_need_segmentation),
    "=",
    len(bbox_ready_images)
)

In [ ]:
# ============================================================
# BATCH SAM SEGMENTATION FOR THE REMAINING 212 IMAGES
# ============================================================

from pathlib import Path
import json
import cv2
import numpy as np
import torch
from ultralytics import SAM
from tqdm.auto import tqdm

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

project_path = Path(r"G:\AIIC")

clean_photos_path = (
    project_path
    / "clean_photos"
)

segmentation_output = (
    project_path
    / "yolo_segmentation"
    / "segmentation_annotations"
)

segmentation_output.mkdir(
    parents=True,
    exist_ok=True
)

image_exts = {
    ".jpg",
    ".jpeg",
    ".png"
}

# ------------------------------------------------------------
# FIND ONLY IMAGES THAT:
# 1. HAVE BBOX JSON
# 2. DO NOT HAVE SEGMENTATION JSON YET
# ------------------------------------------------------------

still_need_segmentation = []

for image_path in clean_photos_path.rglob("*"):

    if not image_path.is_file():
        continue

    if image_path.suffix.lower() not in image_exts:
        continue

    bbox_json = image_path.with_suffix(".json")

    if not bbox_json.exists():
        continue

    seg_json = (
        segmentation_output
        / image_path.parent.name
        / f"{image_path.stem}.json"
    )

    if not seg_json.exists():
        still_need_segmentation.append(image_path)

still_need_segmentation = sorted(
    still_need_segmentation,
    key=lambda p: (
        p.parent.name.lower(),
        p.name.lower()
    )
)

print("=" * 70)
print("SAM BATCH SEGMENTATION")
print("=" * 70)
print("Images to process:", len(still_need_segmentation))

# ------------------------------------------------------------
# DEVICE
# ------------------------------------------------------------

device = 0 if torch.cuda.is_available() else "cpu"

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# LOAD SAM ONCE
# ------------------------------------------------------------

sam = SAM("sam2.1_b.pt")

# ------------------------------------------------------------
# HELPER: LOAD BBOX
# ------------------------------------------------------------

def load_bbox_json(json_path):

    with open(
        json_path,
        "r",
        encoding="utf-8"
    ) as f:
        data = json.load(f)

    boxes = []
    labels = []

    for shape in data.get("shapes", []):

        if shape.get("shape_type") != "rectangle":
            continue

        points = np.array(
            shape.get("points", []),
            dtype=float
        )

        if len(points) < 2:
            continue

        x_min = float(points[:, 0].min())
        y_min = float(points[:, 1].min())
        x_max = float(points[:, 0].max())
        y_max = float(points[:, 1].max())

        if x_max <= x_min or y_max <= y_min:
            continue

        boxes.append(
            [
                x_min,
                y_min,
                x_max,
                y_max
            ]
        )

        labels.append(
            shape.get(
                "label",
                image_path.parent.name
            )
        )

    return boxes, labels


# ------------------------------------------------------------
# HELPER: MASK -> POLYGON
# ------------------------------------------------------------

def mask_to_polygon(
    mask,
    image_width,
    image_height
):

    if mask.shape != (
        image_height,
        image_width
    ):
        mask = cv2.resize(
            mask,
            (
                image_width,
                image_height
            ),
            interpolation=cv2.INTER_NEAREST
        )

    mask_uint8 = (
        (mask > 0.5)
        .astype(np.uint8)
        * 255
    )

    contours, _ = cv2.findContours(
        mask_uint8,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    if not contours:
        return None

    contour = max(
        contours,
        key=cv2.contourArea
    )

    if cv2.contourArea(contour) < 10:
        return None

    epsilon = (
        0.002
        * cv2.arcLength(
            contour,
            True
        )
    )

    approx = cv2.approxPolyDP(
        contour,
        epsilon,
        True
    )

    polygon = (
        approx
        .reshape(-1, 2)
        .astype(float)
        .tolist()
    )

    if len(polygon) < 3:
        return None

    return polygon


# ------------------------------------------------------------
# PROCESS
# ------------------------------------------------------------

generated = 0
review = 0
failed = 0

review_images = []
failed_images = []

# Prevent huge crowded images from sending
# hundreds of prompts to SAM in one shot.
BOX_CHUNK_SIZE = 32

for image_path in tqdm(
    still_need_segmentation,
    desc="SAM segmentation"
):

    try:

        bbox_json = (
            image_path
            .with_suffix(".json")
        )

        boxes, labels = load_bbox_json(
            bbox_json
        )

        # No valid boxes
        if len(boxes) == 0:

            failed += 1

            failed_images.append({
                "class": image_path.parent.name,
                "image": image_path.name,
                "reason": "No valid bbox"
            })

            continue

        # -------------------------
        # READ IMAGE
        # -------------------------

        image_bgr = cv2.imread(
            str(image_path)
        )

        if image_bgr is None:
            raise RuntimeError(
                "Could not read image"
            )

        image_height, image_width = (
            image_bgr.shape[:2]
        )

        # -------------------------
        # SAM
        # -------------------------

        all_masks = []

        for start in range(
            0,
            len(boxes),
            BOX_CHUNK_SIZE
        ):

            chunk_boxes = boxes[
                start:
                start + BOX_CHUNK_SIZE
            ]

            results = sam.predict(
                source=str(image_path),
                bboxes=chunk_boxes,
                device=device,
                verbose=False
            )

            result = results[0]

            if result.masks is None:
                continue

            chunk_masks = (
                result.masks.data
                .cpu()
                .numpy()
            )

            for mask in chunk_masks:
                all_masks.append(mask)

        # -------------------------
        # NO MASK
        # -------------------------

        if len(all_masks) == 0:

            failed += 1

            failed_images.append({
                "class": image_path.parent.name,
                "image": image_path.name,
                "reason": "SAM generated no masks"
            })

            continue

        # -------------------------
        # MASK -> POLYGON
        # -------------------------

        shapes = []

        usable_count = min(
            len(all_masks),
            len(labels)
        )

        for i in range(
            usable_count
        ):

            polygon = mask_to_polygon(
                all_masks[i],
                image_width,
                image_height
            )

            if polygon is None:
                continue

            shapes.append({
                "label": labels[i],
                "points": polygon,
                "group_id": None,
                "description": "",
                "shape_type": "polygon",
                "flags": {}
            })

        # No usable polygon
        if len(shapes) == 0:

            failed += 1

            failed_images.append({
                "class": image_path.parent.name,
                "image": image_path.name,
                "reason": "Masks found but polygon conversion failed"
            })

            continue

        # -------------------------
        # SAVE
        # -------------------------

        class_output = (
            segmentation_output
            / image_path.parent.name
        )

        class_output.mkdir(
            parents=True,
            exist_ok=True
        )

        seg_json_path = (
            class_output
            / f"{image_path.stem}.json"
        )

        json_data = {
            "version": "5.0.1",
            "flags": {},
            "shapes": shapes,
            "imagePath": image_path.name,
            "imageData": None,
            "imageHeight": image_height,
            "imageWidth": image_width
        }

        with open(
            seg_json_path,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                json_data,
                f,
                indent=2,
                ensure_ascii=False
            )

        generated += 1

        # -------------------------
        # REVIEW FLAG
        # -------------------------

        if (
            len(all_masks) != len(boxes)
            or len(shapes) != len(boxes)
        ):

            review += 1

            review_images.append({
                "class": image_path.parent.name,
                "image": image_path.name,
                "bbox_count": len(boxes),
                "mask_count": len(all_masks),
                "polygon_count": len(shapes)
            })

    except Exception as e:

        failed += 1

        failed_images.append({
            "class": image_path.parent.name,
            "image": image_path.name,
            "reason": str(e)
        })

# ------------------------------------------------------------
# SAVE REPORT
# ------------------------------------------------------------

report_path = (
    project_path
    / "yolo_segmentation"
    / "sam_remaining_212_report.json"
)

report = {
    "requested": len(
        still_need_segmentation
    ),
    "generated": generated,
    "review": review,
    "failed": failed,
    "review_images": review_images,
    "failed_images": failed_images
}

with open(
    report_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        report,
        f,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# FINAL SUMMARY
# ------------------------------------------------------------

print("\n")
print("=" * 70)
print("SAM BATCH COMPLETE")
print("=" * 70)

print(
    "Requested :",
    len(still_need_segmentation)
)

print(
    "Generated :",
    generated
)

print(
    "Review    :",
    review
)

print(
    "Failed    :",
    failed
)

print("\nReport:")
print(report_path)

if failed_images:

    print("\nFAILED IMAGES:")

    for item in failed_images:

        print(
            f"[{item['class']}] "
            f"{item['image']}"
        )

        print(
            "   →",
            item["reason"]
        )

In [ ]:
# ============================================================
# SHOW SAM REVIEW IMAGES
# ============================================================

import json
from pathlib import Path

report_path = Path(
    r"G:\AIIC\yolo_segmentation\sam_remaining_212_report.json"
)

with open(report_path, "r", encoding="utf-8") as f:
    report = json.load(f)

review_images = report.get("review_images", [])

print("=" * 65)
print("SAM REVIEW REQUIRED")
print("=" * 65)

print("Total review:", len(review_images))

for i, item in enumerate(review_images, start=1):
    print(f"\n{i}. [{item['class']}] {item['image']}")
    print("   BBox    :", item["bbox_count"])
    print("   Masks   :", item["mask_count"])
    print("   Polygons:", item["polygon_count"])
    

In [ ]:
# ============================================================
# RETRY ONE REVIEW IMAGE — ONE BBOX AT A TIME
# ============================================================

from pathlib import Path
import json
import cv2
import numpy as np
import torch
from ultralytics import SAM

project_path = Path(r"G:\AIIC")

image_path = (
    project_path
    / "clean_photos"
    / "E004_Breadboard"
    / "WhatsApp Image 2026-08-12 at 11.07.17 AM.jpg"
)

bbox_json_path = image_path.with_suffix(".json")

segmentation_output = (
    project_path
    / "yolo_segmentation"
    / "segmentation_annotations"
)

seg_json_path = (
    segmentation_output
    / image_path.parent.name
    / f"{image_path.stem}.json"
)

# ------------------------------------------------------------
# LOAD BBOX JSON
# ------------------------------------------------------------

with open(bbox_json_path, "r", encoding="utf-8") as f:
    bbox_data = json.load(f)

boxes = []
labels = []

for shape in bbox_data.get("shapes", []):

    if shape.get("shape_type") != "rectangle":
        continue

    points = np.array(shape["points"], dtype=float)

    x_min = float(points[:, 0].min())
    y_min = float(points[:, 1].min())
    x_max = float(points[:, 0].max())
    y_max = float(points[:, 1].max())

    boxes.append([
        x_min,
        y_min,
        x_max,
        y_max
    ])

    labels.append(shape["label"])

print("BBox count:", len(boxes))

# ------------------------------------------------------------
# IMAGE
# ------------------------------------------------------------

image_bgr = cv2.imread(str(image_path))

if image_bgr is None:
    raise RuntimeError("Cannot read image")

image_height, image_width = image_bgr.shape[:2]

# ------------------------------------------------------------
# SAM
# ------------------------------------------------------------

device = 0 if torch.cuda.is_available() else "cpu"

sam = SAM("sam2.1_b.pt")

# ------------------------------------------------------------
# MASK -> POLYGON
# ------------------------------------------------------------

def mask_to_polygon(mask):

    if mask.shape != (image_height, image_width):

        mask = cv2.resize(
            mask,
            (image_width, image_height),
            interpolation=cv2.INTER_NEAREST
        )

    mask_uint8 = (
        (mask > 0.5)
        .astype(np.uint8)
        * 255
    )

    contours, _ = cv2.findContours(
        mask_uint8,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    if not contours:
        return None

    contour = max(
        contours,
        key=cv2.contourArea
    )

    if cv2.contourArea(contour) < 10:
        return None

    epsilon = (
        0.002
        * cv2.arcLength(contour, True)
    )

    approx = cv2.approxPolyDP(
        contour,
        epsilon,
        True
    )

    polygon = (
        approx
        .reshape(-1, 2)
        .astype(float)
        .tolist()
    )

    if len(polygon) < 3:
        return None

    return polygon

# ------------------------------------------------------------
# RUN EACH BBOX SEPARATELY
# ------------------------------------------------------------

shapes = []
failed_boxes = []

for i, (box, label) in enumerate(
    zip(boxes, labels),
    start=1
):

    try:

        results = sam.predict(
            source=str(image_path),
            bboxes=[box],
            device=device,
            verbose=False
        )

        result = results[0]

        if result.masks is None:
            failed_boxes.append(i)
            continue

        masks = (
            result.masks.data
            .cpu()
            .numpy()
        )

        if len(masks) == 0:
            failed_boxes.append(i)
            continue

        polygon = mask_to_polygon(
            masks[0]
        )

        if polygon is None:
            failed_boxes.append(i)
            continue

        shapes.append({
            "label": label,
            "points": polygon,
            "group_id": None,
            "description": "",
            "shape_type": "polygon",
            "flags": {}
        })

    except Exception:
        failed_boxes.append(i)

# ------------------------------------------------------------
# SAVE / OVERWRITE SEGMENTATION JSON
# ------------------------------------------------------------

seg_json_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

json_data = {
    "version": "5.0.1",
    "flags": {},
    "shapes": shapes,
    "imagePath": image_path.name,
    "imageData": None,
    "imageHeight": image_height,
    "imageWidth": image_width
}

with open(
    seg_json_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        json_data,
        f,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("BREADBOARD RETRY RESULT")
print("=" * 60)

print("BBox     :", len(boxes))
print("Polygons :", len(shapes))
print("Failed   :", len(failed_boxes))

if failed_boxes:
    print("Failed bbox numbers:", failed_boxes)

print("\nSaved:")
print(seg_json_path)

In [ ]:
# ============================================================
# SHOW FAILED BREADBOARD BBOXES
# ============================================================

from pathlib import Path
import json
import cv2
import numpy as np
import matplotlib.pyplot as plt

project_path = Path(r"G:\AIIC")

image_path = (
    project_path
    / "clean_photos"
    / "E004_Breadboard"
    / "WhatsApp Image 2026-08-12 at 11.07.17 AM.jpg"
)

bbox_json_path = image_path.with_suffix(".json")

failed_boxes = {13, 14, 15, 16, 17, 18, 19}

# Load image
image_bgr = cv2.imread(str(image_path))
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

# Load bbox JSON
with open(bbox_json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

boxes = []

for shape in data.get("shapes", []):

    if shape.get("shape_type") != "rectangle":
        continue

    points = np.array(shape["points"], dtype=float)

    x1 = int(points[:, 0].min())
    y1 = int(points[:, 1].min())
    x2 = int(points[:, 0].max())
    y2 = int(points[:, 1].max())

    boxes.append((x1, y1, x2, y2))

# Plot
fig, ax = plt.subplots(figsize=(10, 16))

ax.imshow(image_rgb)

for i, (x1, y1, x2, y2) in enumerate(boxes, start=1):

    linewidth = 4 if i in failed_boxes else 1

    rect = plt.Rectangle(
        (x1, y1),
        x2 - x1,
        y2 - y1,
        fill=False,
        linewidth=linewidth
    )

    ax.add_patch(rect)

    ax.text(
        x1,
        y1,
        str(i),
        fontsize=12,
        bbox=dict(facecolor="white", alpha=0.8)
    )

ax.set_title(
    "Breadboard BBoxes — 13 to 19 FAILED SAM"
)

ax.axis("off")
plt.show()

In [ ]:
# ============================================================
# FINAL SEGMENTATION CHECKER
# ============================================================

from pathlib import Path

project_path = Path(r"G:\AIIC")
clean_photos_path = project_path / "clean_photos"

segmentation_output = (
    project_path
    / "yolo_segmentation"
    / "segmentation_annotations"
)

image_exts = {".jpg", ".jpeg", ".png"}

total_images = 0
with_segmentation = 0
without_segmentation = []

for img in clean_photos_path.rglob("*"):

    if not img.is_file():
        continue

    if img.suffix.lower() not in image_exts:
        continue

    total_images += 1

    seg_json = (
        segmentation_output
        / img.parent.name
        / f"{img.stem}.json"
    )

    if seg_json.exists():
        with_segmentation += 1
    else:
        without_segmentation.append(img)

print("=" * 65)
print("FINAL SEGMENTATION STATUS")
print("=" * 65)

print("Total images             :", total_images)
print("With segmentation JSON   :", with_segmentation)
print("Without segmentation JSON:", len(without_segmentation))

print("\nExpected:")
print("2726 segmentation")
print("4 intentional exclusions")
print("2730 total")

if without_segmentation:
    print("\nWITHOUT SEGMENTATION:")
    for img in without_segmentation:
        print(f"[{img.parent.name}] {img.name}")

In [ ]:
# ============================================================
# FINAL LABEL AUDIT — WITH ALL PATCHES INCLUDED
# ============================================================

from pathlib import Path
import json
import re
from collections import Counter

project_path = Path(r"G:\AIIC")

clean_photos_path = (
    project_path
    / "clean_photos"
)

segmentation_output = (
    project_path
    / "yolo_segmentation"
    / "segmentation_annotations"
)

# ------------------------------------------------------------
# CANONICAL CLASSES
# ------------------------------------------------------------

class_names = sorted(
    [
        p.name
        for p in clean_photos_path.iterdir()
        if p.is_dir()
    ],
    key=str.lower
)

print("Canonical classes:", len(class_names))


def normalize_text(text):

    text = str(text).lower().strip()

    text = re.sub(
        r"^[a-z]\d{3}[_\-\s]*",
        "",
        text
    )

    text = (
        text
        .replace("_", " ")
        .replace("-", " ")
        .replace("×", "x")
        .replace("ω", "ohm")
        .replace("Ω", "ohm")
    )

    text = re.sub(
        r"[^a-z0-9]+",
        "",
        text
    )

    return text


# ------------------------------------------------------------
# CANONICAL LOOKUP
# ------------------------------------------------------------

canonical_lookup = {
    normalize_text(name): name
    for name in class_names
}

# ------------------------------------------------------------
# ALL KNOWN ALIASES
# ------------------------------------------------------------

aliases = {

    normalize_text("Arduino Uno"):
        "E001_Arduino Uno",

    normalize_text("Arduino Nano"):
        "E002_Arduino Nano",

    normalize_text("Motor Shield"):
        "E003_Motor Shield",

    normalize_text("Bread Board"):
        "E004_Breadboard",

    normalize_text("Breadboard"):
        "E004_Breadboard",

    normalize_text("Gas Sensor"):
        "E005_Gas Sensor MQ6",

    normalize_text("Mini Micro Metal Gear"):
        "E008_N20 Mini Micro Metal Gear",

    normalize_text("Motor 3V"):
        "E009_DC Motor 3V",

    normalize_text("DC Motor"):
        "E009_DC Motor 3V",

    normalize_text("Temperature and Humidity"):
        "E011_DHT11 Temperature and humidity",

    normalize_text("LED Red"):
        "E018_LED Red",

    normalize_text("LED Blue"):
        "E019_LED Blue",

    normalize_text("Heart Rate and Pulse Oximetry Sensor"):
        "E020_MAX30102 Heart Rate and Pulse Oximetry Sensor",

    normalize_text("Soil Moisture Level Sensor"):
        "E021_Soil Moisture Level Sensor",

    normalize_text("Light Dependent Resistor (LDR)"):
        "E024_Light Dependent Resistor (LDR)",

    normalize_text("Resistor 220Ω"):
        "E025_Resistor 220Ω",

    normalize_text("LCD Display"):
        "E030_16×2 LCD Display",

    # ========================================================
    # PATCHED MEMBER LABELS
    # ========================================================

    normalize_text("A4 coloured paper"):
        "C014_A4 colored paper",

    normalize_text("Battery Holder"):
        "E016_CR2032 battery holder",

    normalize_text("d"):
        "C002_paper cup",

    normalize_text("Lithium Cloride"):
        "L025_Lithium Chloride",

    normalize_text("Measure Tape"):
        "T005_MeasuringTape",

    normalize_text("Micro Metal Motor"):
        "E010_N20 Micro Metal Gear Motor",

    normalize_text("RFID Card Reader"):
        "E017_RC522 RFID Card Reader",

    normalize_text("Ultrasonic Sensor"):
        "E007_HC-SR04 Ultrasonic Sensor",
}

# ------------------------------------------------------------
# COLLECT RAW LABELS
# ------------------------------------------------------------

raw_labels = Counter()

for json_path in segmentation_output.rglob("*.json"):

    with open(
        json_path,
        "r",
        encoding="utf-8"
    ) as f:
        data = json.load(f)

    for shape in data.get("shapes", []):

        if shape.get("shape_type") != "polygon":
            continue

        label = str(
            shape.get("label", "")
        ).strip()

        if label:
            raw_labels[label] += 1

# ------------------------------------------------------------
# AUDIT
# ------------------------------------------------------------

mapped = {}
unmatched = {}

for raw_label, count in raw_labels.items():

    key = normalize_text(raw_label)

    if key in canonical_lookup:

        mapped[raw_label] = (
            canonical_lookup[key]
        )

    elif key in aliases:

        mapped[raw_label] = (
            aliases[key]
        )

    else:

        unmatched[raw_label] = count


print("\n" + "=" * 65)
print("FINAL LABEL AUDIT")
print("=" * 65)

print("Unique raw labels :", len(raw_labels))
print("Mapped labels     :", len(mapped))
print("Unmatched labels  :", len(unmatched))

if unmatched:

    print("\nUNMATCHED LABELS:")

    for label in sorted(
        unmatched,
        key=str.lower
    ):
        print(
            f"{unmatched[label]:5} | "
            f"{label}"
        )

else:

    print("\n✅ ALL 116 RAW LABELS MAPPED")
    print("✅ 109 CANONICAL CLASSES")
    print("✅ READY FOR YOLO CONVERSION")

In [ ]:
# ============================================================
# TRACE UNMATCHED LABELS + SUGGEST CANONICAL CLASS
# ============================================================

import json
import difflib
from collections import defaultdict
from pathlib import Path

project_path = Path(r"G:\AIIC")

clean_photos_path = project_path / "clean_photos"

segmentation_output = (
    project_path
    / "yolo_segmentation"
    / "segmentation_annotations"
)

# Canonical classes = actual clean_photos folder names
class_names = sorted(
    [p.name for p in clean_photos_path.iterdir() if p.is_dir()],
    key=str.lower
)

unmatched_names = {
    "A4 coloured paper",
    "Battery Holder",
    "d",
    "Lithium Cloride",
    "Measure Tape",
    "Micro Metal Motor",
    "RFID Card Reader",
    "Ultrasonic Sensor",
}

locations = defaultdict(list)

for json_path in segmentation_output.rglob("*.json"):

    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    for shape in data.get("shapes", []):

        raw_label = str(shape.get("label", "")).strip()

        if raw_label in unmatched_names:
            locations[raw_label].append({
                "folder": json_path.parent.name,
                "file": json_path.name
            })


print("=" * 75)
print("UNMATCHED LABEL INVESTIGATION")
print("=" * 75)

for label in sorted(unmatched_names, key=str.lower):

    print(f"\nLABEL: {label}")
    print("-" * 75)

    # Where does this label occur?
    locs = locations.get(label, [])

    folders = sorted(set(x["folder"] for x in locs))

    print("Count   :", len(locs))
    print("Folders :", folders)

    # Fuzzy canonical suggestions
    suggestions = difflib.get_close_matches(
        label,
        class_names,
        n=5,
        cutoff=0.15
    )

    print("Possible canonical classes:")

    for s in suggestions:
        print("   →", s)

    # show a few files
    if locs:
        print("Example files:")

        for x in locs[:3]:
            print(
                f"   [{x['folder']}] "
                f"{x['file']}"
            )

In [ ]:
# ============================================================
# PATCH THE 8 UNMATCHED LABELS
# ============================================================

manual_aliases = {
    "A4 coloured paper": "C014_A4 colored paper",
    "Battery Holder": "E016_CR2032 battery holder",
    "d": "C002_paper cup",
    "Lithium Cloride": "L025_Lithium Chloride",
    "Measure Tape": "T005_MeasuringTape",
    "Micro Metal Motor": "E010_N20 Micro Metal Gear Motor",
    "RFID Card Reader": "E017_RC522 RFID Card Reader",
    "Ultrasonic Sensor": "E007_HC-SR04 Ultrasonic Sensor",
}

# Add into existing alias system
for raw_label, canonical_class in manual_aliases.items():
    aliases[normalize_text(raw_label)] = canonical_class

print("=" * 65)
print("MANUAL LABEL PATCH")
print("=" * 65)

for raw_label, canonical in manual_aliases.items():
    print(f"{raw_label:<30} -> {canonical}")

print("\nPatched:", len(manual_aliases))

In [51]:
# ============================================================
# FIND THE 3 SEGMENTATION JSONs WITH MISSING SOURCE IMAGE
# ============================================================

from pathlib import Path
import json

project_path = Path(r"G:\AIIC")

clean_photos_path = project_path / "clean_photos"

segmentation_output = (
    project_path
    / "yolo_segmentation"
    / "segmentation_annotations"
)

problem_files = []

all_seg_jsons = list(
    segmentation_output.rglob("*.json")
)

for seg_json in all_seg_jsons:

    class_folder = seg_json.parent.name

    with open(
        seg_json,
        "r",
        encoding="utf-8"
    ) as f:
        data = json.load(f)

    image_name = data.get("imagePath")

    if not image_name:
        problem_files.append({
            "json": seg_json,
            "imagePath": None,
            "reason": "imagePath missing"
        })
        continue

    expected_image = (
        clean_photos_path
        / class_folder
        / image_name
    )

    if not expected_image.exists():
        problem_files.append({
            "json": seg_json,
            "imagePath": image_name,
            "reason": "source image not found"
        })


print("=" * 70)
print("SEGMENTATION SOURCE IMAGE CHECK")
print("=" * 70)

print("Segmentation JSONs :", len(all_seg_jsons))
print("Problems           :", len(problem_files))

for i, item in enumerate(problem_files, start=1):

    print(f"\n{i}. {item['json']}")
    print("   imagePath :", item["imagePath"])
    print("   reason    :", item["reason"])

SEGMENTATION SOURCE IMAGE CHECK
Segmentation JSONs : 2726
Problems           : 3

1. G:\AIIC\yolo_segmentation\segmentation_annotations\E001_Arduino Uno\WhatsApp Image 2026-08-11 at 6.07.43 PM.json
   imagePath : ..\..\..\clean_photos\E001_Arduino Uno\WhatsApp Image 2026-08-11 at 6.07.43 PM.jpg
   reason    : source image not found

2. G:\AIIC\yolo_segmentation\segmentation_annotations\E001_Arduino Uno\WhatsApp Image 2026-08-11 at 6.07.44 PM (1).json
   imagePath : ..\..\..\clean_photos\E001_Arduino Uno\WhatsApp Image 2026-08-11 at 6.07.44 PM (1).jpg
   reason    : source image not found

3. G:\AIIC\yolo_segmentation\segmentation_annotations\E001_Arduino Uno\WhatsApp Image 2026-08-11 at 6.07.44 PM.json
   imagePath : ..\..\..\clean_photos\E001_Arduino Uno\WhatsApp Image 2026-08-11 at 6.07.44 PM.jpg
   reason    : source image not found


In [52]:
# ============================================================
# FIX OLD SEGMENTATION JSON imagePath
# ============================================================

from pathlib import Path
import json

project_path = Path(r"G:\AIIC")

segmentation_output = (
    project_path
    / "yolo_segmentation"
    / "segmentation_annotations"
)

fixed = 0

for seg_json in segmentation_output.rglob("*.json"):

    with open(seg_json, "r", encoding="utf-8") as f:
        data = json.load(f)

    image_path_value = data.get("imagePath")

    if not image_path_value:
        continue

    # Convert old paths:
    # ..\..\..\clean_photos\class\image.jpg
    # into:
    # image.jpg
    image_name = (
        str(image_path_value)
        .replace("\\", "/")
        .split("/")[-1]
    )

    if image_name != image_path_value:

        data["imagePath"] = image_name

        with open(seg_json, "w", encoding="utf-8") as f:
            json.dump(
                data,
                f,
                indent=2,
                ensure_ascii=False
            )

        print(f"FIXED: {seg_json.name}")
        print(f"   OLD: {image_path_value}")
        print(f"   NEW: {image_name}\n")

        fixed += 1


print("=" * 60)
print("Fixed JSONs:", fixed)

FIXED: WhatsApp Image 2026-08-11 at 6.07.43 PM.json
   OLD: ..\..\..\clean_photos\E001_Arduino Uno\WhatsApp Image 2026-08-11 at 6.07.43 PM.jpg
   NEW: WhatsApp Image 2026-08-11 at 6.07.43 PM.jpg

FIXED: WhatsApp Image 2026-08-11 at 6.07.44 PM (1).json
   OLD: ..\..\..\clean_photos\E001_Arduino Uno\WhatsApp Image 2026-08-11 at 6.07.44 PM (1).jpg
   NEW: WhatsApp Image 2026-08-11 at 6.07.44 PM (1).jpg

FIXED: WhatsApp Image 2026-08-11 at 6.07.44 PM.json
   OLD: ..\..\..\clean_photos\E001_Arduino Uno\WhatsApp Image 2026-08-11 at 6.07.44 PM.jpg
   NEW: WhatsApp Image 2026-08-11 at 6.07.44 PM.jpg

Fixed JSONs: 3


In [54]:
# ============================================================
# BUILD FINAL YOLO SEGMENTATION DATASET
# ============================================================

from pathlib import Path
from collections import defaultdict
import json
import shutil
import random
import re

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

project_path = Path(r"G:\AIIC")
clean_photos_path = project_path / "clean_photos"

segmentation_output = (
    project_path
    / "yolo_segmentation"
    / "segmentation_annotations"
)

yolo_dataset = (
    project_path
    / "yolo_segmentation"
    / "dataset"
)

# Fresh rebuild
if yolo_dataset.exists():
    shutil.rmtree(yolo_dataset)

for split in ["train", "val", "test"]:
    (yolo_dataset / "images" / split).mkdir(parents=True, exist_ok=True)
    (yolo_dataset / "labels" / split).mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 109 CANONICAL CLASSES
# ------------------------------------------------------------

class_names = sorted(
    [p.name for p in clean_photos_path.iterdir() if p.is_dir()],
    key=str.lower
)

class_to_id = {
    name: i
    for i, name in enumerate(class_names)
}

canonical_lookup = {
    normalize_text(name): name
    for name in class_names
}

def get_canonical_class(raw_label):

    key = normalize_text(raw_label)

    if key in canonical_lookup:
        return canonical_lookup[key]

    if key in aliases:
        return aliases[key]

    return None

# ------------------------------------------------------------
# COLLECT SEGMENTATION SAMPLES
# ------------------------------------------------------------

samples = []
missing_images = []

for seg_json in segmentation_output.rglob("*.json"):

    class_folder = seg_json.parent.name

    with open(seg_json, "r", encoding="utf-8") as f:
        data = json.load(f)

    image_name = data.get("imagePath")

    if not image_name:
        missing_images.append((seg_json, "No imagePath"))
        continue

    image_path = (
        clean_photos_path
        / class_folder
        / image_name
    )

    if not image_path.exists():
        missing_images.append((seg_json, image_name))
        continue

    samples.append((image_path, seg_json))

print("Segmentation samples found:", len(samples))

# ------------------------------------------------------------
# STRATIFIED SPLIT BY SOURCE CLASS FOLDER
# 70 / 15 / 15
# ------------------------------------------------------------

random.seed(42)

by_class = defaultdict(list)

for image_path, seg_json in samples:
    by_class[image_path.parent.name].append(
        (image_path, seg_json)
    )

split_samples = {
    "train": [],
    "val": [],
    "test": []
}

for class_name, items in by_class.items():

    random.shuffle(items)

    n = len(items)

    if n >= 3:
        n_train = max(1, round(n * 0.70))
        n_val = max(1, round(n * 0.15))

        # Always leave at least 1 test image
        if n_train + n_val >= n:
            n_train = n - 2
            n_val = 1

    elif n == 2:
        n_train = 1
        n_val = 1

    else:
        n_train = 1
        n_val = 0

    split_samples["train"].extend(
        items[:n_train]
    )

    split_samples["val"].extend(
        items[n_train:n_train + n_val]
    )

    split_samples["test"].extend(
        items[n_train + n_val:]
    )

# ------------------------------------------------------------
# JSON POLYGON -> YOLO SEGMENTATION TXT
# ------------------------------------------------------------

conversion_errors = []
empty_annotations = []

written = {
    "train": 0,
    "val": 0,
    "test": 0
}

for split, items in split_samples.items():

    for image_path, seg_json in items:

        with open(seg_json, "r", encoding="utf-8") as f:
            data = json.load(f)

        width = data.get("imageWidth")
        height = data.get("imageHeight")

        if not width or not height:
            conversion_errors.append(
                (str(seg_json), "Missing image size")
            )
            continue

        yolo_lines = []

        for shape in data.get("shapes", []):

            if shape.get("shape_type") != "polygon":
                continue

            raw_label = str(
                shape.get("label", "")
            ).strip()

            canonical_class = get_canonical_class(
                raw_label
            )

            if canonical_class is None:

                conversion_errors.append(
                    (str(seg_json), raw_label)
                )

                continue

            points = shape.get("points", [])

            if len(points) < 3:
                continue

            class_id = class_to_id[
                canonical_class
            ]

            coords = []

            for x, y in points:

                nx = max(
                    0.0,
                    min(1.0, float(x) / width)
                )

                ny = max(
                    0.0,
                    min(1.0, float(y) / height)
                )

                coords.extend([nx, ny])

            yolo_line = (
                str(class_id)
                + " "
                + " ".join(
                    f"{v:.6f}"
                    for v in coords
                )
            )

            yolo_lines.append(yolo_line)

        # Don't silently turn target image into background
        if len(yolo_lines) == 0:

            empty_annotations.append(
                str(seg_json)
            )

            continue

        # ------------------------------------
        # UNIQUE OUTPUT NAME
        # avoids duplicate WhatsApp filenames
        # ------------------------------------

        safe_class = re.sub(
            r"[^A-Za-z0-9]+",
            "_",
            image_path.parent.name
        ).strip("_")

        output_stem = (
            f"{safe_class}__{image_path.stem}"
        )

        output_image = (
            yolo_dataset
            / "images"
            / split
            / f"{output_stem}{image_path.suffix.lower()}"
        )

        output_label = (
            yolo_dataset
            / "labels"
            / split
            / f"{output_stem}.txt"
        )

        shutil.copy2(
            image_path,
            output_image
        )

        with open(
            output_label,
            "w",
            encoding="utf-8"
        ) as f:
            f.write("\n".join(yolo_lines))

        written[split] += 1

# ------------------------------------------------------------
# CREATE data.yaml
# ------------------------------------------------------------

yaml_path = yolo_dataset / "data.yaml"

with open(yaml_path, "w", encoding="utf-8") as f:

    f.write(
        f"path: {yolo_dataset.as_posix()}\n"
    )

    f.write("train: images/train\n")
    f.write("val: images/val\n")
    f.write("test: images/test\n\n")

    f.write("names:\n")

    for class_id, class_name in enumerate(class_names):

        safe_name = class_name.replace('"', '\\"')

        f.write(
            f'  {class_id}: "{safe_name}"\n'
        )

# ------------------------------------------------------------
# FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("YOLO SEGMENTATION DATASET COMPLETE")
print("=" * 70)

print("Train :", written["train"])
print("Val   :", written["val"])
print("Test  :", written["test"])

print(
    "Total :",
    sum(written.values())
)

print("Classes:", len(class_names))

print(
    "Conversion errors:",
    len(conversion_errors)
)

print(
    "Empty annotations:",
    len(empty_annotations)
)

print(
    "Missing source images:",
    len(missing_images)
)

print("\nDataset:")
print(yolo_dataset)

print("\nYAML:")
print(yaml_path)

Segmentation samples found: 2726

YOLO SEGMENTATION DATASET COMPLETE
Train : 1907
Val   : 403
Test  : 413
Total : 2723
Classes: 109
Conversion errors: 0
Empty annotations: 3
Missing source images: 0

Dataset:
G:\AIIC\yolo_segmentation\dataset

YAML:
G:\AIIC\yolo_segmentation\dataset\data.yaml


In [55]:
# ============================================================
# FIND THE 3 SAMPLES LOST DURING SPLIT
# ============================================================

from collections import Counter

print("=" * 70)
print("SPLIT INTEGRITY CHECK")
print("=" * 70)

print("Samples before split :", len(samples))

grouped_total = sum(
    len(items)
    for items in by_class.values()
)

split_total = sum(
    len(items)
    for items in split_samples.values()
)

print("Grouped by class     :", grouped_total)
print("After split          :", split_total)
print("Difference           :", len(samples) - split_total)

# ------------------------------------------------------------
# Compare using exact image + segmentation JSON pair
# ------------------------------------------------------------

before = Counter(
    (str(img), str(js))
    for img, js in samples
)

after = Counter(
    (str(img), str(js))
    for split_items in split_samples.values()
    for img, js in split_items
)

lost = before - after

print("\n" + "=" * 70)
print("LOST SAMPLES")
print("=" * 70)

print("Total lost:", sum(lost.values()))

for i, ((img, js), count) in enumerate(
    lost.items(),
    start=1
):
    print(f"\n{i}.")
    print("Image:", img)
    print("JSON :", js)
    print("Count:", count)

SPLIT INTEGRITY CHECK
Samples before split : 2726
Grouped by class     : 2726
After split          : 2726
Difference           : 0

LOST SAMPLES
Total lost: 0


In [56]:
# ============================================================
# FIND THE 3 NOT WRITTEN TO YOLO DATASET
# ============================================================

from pathlib import Path
import re

print("=" * 70)
print("YOLO WRITE INTEGRITY CHECK")
print("=" * 70)

missing_written = []

for split, items in split_samples.items():

    print(
        f"{split.upper():5} | "
        f"split samples = {len(items)} | "
        f"written counter = {written[split]}"
    )

    for image_path, seg_json in items:

        safe_class = re.sub(
            r"[^A-Za-z0-9]+",
            "_",
            image_path.parent.name
        ).strip("_")

        output_stem = (
            f"{safe_class}__{image_path.stem}"
        )

        expected_image = (
            yolo_dataset
            / "images"
            / split
            / f"{output_stem}{image_path.suffix.lower()}"
        )

        expected_label = (
            yolo_dataset
            / "labels"
            / split
            / f"{output_stem}.txt"
        )

        if not expected_image.exists() or not expected_label.exists():

            missing_written.append({
                "split": split,
                "image": image_path,
                "json": seg_json,
                "image_exists": expected_image.exists(),
                "label_exists": expected_label.exists(),
            })


print("\n" + "=" * 70)
print("NOT WRITTEN")
print("=" * 70)

print("Total:", len(missing_written))

for i, item in enumerate(missing_written, start=1):

    print(f"\n{i}. [{item['split']}]")
    print("Image :", item["image"])
    print("JSON  :", item["json"])
    print("YOLO image exists:", item["image_exists"])
    print("YOLO label exists:", item["label_exists"])

YOLO WRITE INTEGRITY CHECK
TRAIN | split samples = 1909 | written counter = 1907
VAL   | split samples = 403 | written counter = 403
TEST  | split samples = 414 | written counter = 413

NOT WRITTEN
Total: 3

1. [train]
Image : G:\AIIC\clean_photos\E001_Arduino Uno\WhatsApp Image 2026-08-11 at 6.07.44 PM (1).jpg
JSON  : G:\AIIC\yolo_segmentation\segmentation_annotations\E001_Arduino Uno\WhatsApp Image 2026-08-11 at 6.07.44 PM (1).json
YOLO image exists: False
YOLO label exists: False

2. [train]
Image : G:\AIIC\clean_photos\E001_Arduino Uno\WhatsApp Image 2026-08-11 at 6.07.44 PM.jpg
JSON  : G:\AIIC\yolo_segmentation\segmentation_annotations\E001_Arduino Uno\WhatsApp Image 2026-08-11 at 6.07.44 PM.json
YOLO image exists: False
YOLO label exists: False

3. [test]
Image : G:\AIIC\clean_photos\E001_Arduino Uno\WhatsApp Image 2026-08-11 at 6.07.43 PM.jpg
JSON  : G:\AIIC\yolo_segmentation\segmentation_annotations\E001_Arduino Uno\WhatsApp Image 2026-08-11 at 6.07.43 PM.json
YOLO image exists

In [57]:
# ============================================================
# PATCH THE 3 LEGACY ARDUINO SAMPLES DIRECTLY
# ============================================================

import json
import re
import shutil
import cv2

print("=" * 70)
print("PATCHING 3 LEGACY ARDUINO SAMPLES")
print("=" * 70)

patched = 0
failed_patch = []

for item in missing_written:

    split = item["split"]
    image_path = item["image"]
    seg_json = item["json"]

    print(f"\n[{split.upper()}] {image_path.name}")

    # --------------------------------------------------------
    # READ IMAGE FOR TRUE WIDTH / HEIGHT
    # --------------------------------------------------------

    img = cv2.imread(str(image_path))

    if img is None:
        failed_patch.append((image_path.name, "Cannot read image"))
        print("❌ Cannot read image")
        continue

    image_height, image_width = img.shape[:2]

    # --------------------------------------------------------
    # READ SEGMENTATION JSON
    # --------------------------------------------------------

    with open(seg_json, "r", encoding="utf-8") as f:
        data = json.load(f)

    shapes = data.get("shapes", [])

    print("Shapes in JSON:", len(shapes))
    print(
        "Shape types:",
        sorted(set(str(s.get("shape_type")) for s in shapes))
    )

    yolo_lines = []

    # --------------------------------------------------------
    # CONVERT POLYGONS
    # --------------------------------------------------------

    for shape in shapes:

        if shape.get("shape_type") != "polygon":
            continue

        points = shape.get("points", [])

        if len(points) < 3:
            continue

        raw_label = str(
            shape.get("label", "")
        ).strip()

        canonical_class = get_canonical_class(raw_label)

        if canonical_class is None:
            print("❌ Unknown label:", raw_label)
            continue

        class_id = class_to_id[canonical_class]

        coords = []

        for x, y in points:

            nx = max(
                0.0,
                min(1.0, float(x) / image_width)
            )

            ny = max(
                0.0,
                min(1.0, float(y) / image_height)
            )

            coords.extend([nx, ny])

        yolo_lines.append(
            str(class_id)
            + " "
            + " ".join(f"{v:.6f}" for v in coords)
        )

    print("Valid polygons:", len(yolo_lines))

    if len(yolo_lines) == 0:

        failed_patch.append(
            (
                image_path.name,
                "No valid polygon"
            )
        )

        print("❌ No valid polygon")
        continue

    # --------------------------------------------------------
    # SAME UNIQUE OUTPUT NAMING
    # --------------------------------------------------------

    safe_class = re.sub(
        r"[^A-Za-z0-9]+",
        "_",
        image_path.parent.name
    ).strip("_")

    output_stem = (
        f"{safe_class}__{image_path.stem}"
    )

    output_image = (
        yolo_dataset
        / "images"
        / split
        / f"{output_stem}{image_path.suffix.lower()}"
    )

    output_label = (
        yolo_dataset
        / "labels"
        / split
        / f"{output_stem}.txt"
    )

    shutil.copy2(
        image_path,
        output_image
    )

    with open(
        output_label,
        "w",
        encoding="utf-8"
    ) as f:
        f.write("\n".join(yolo_lines))

    patched += 1

    print("✅ WRITTEN")


print("\n" + "=" * 70)
print("PATCH RESULT")
print("=" * 70)

print("Patched :", patched)
print("Failed  :", len(failed_patch))

if failed_patch:
    print("\nFailures:")
    for name, reason in failed_patch:
        print(name, "→", reason)

PATCHING 3 LEGACY ARDUINO SAMPLES

[TRAIN] WhatsApp Image 2026-08-11 at 6.07.44 PM (1).jpg
Shapes in JSON: 1
Shape types: ['rectangle']
Valid polygons: 0
❌ No valid polygon

[TRAIN] WhatsApp Image 2026-08-11 at 6.07.44 PM.jpg
Shapes in JSON: 1
Shape types: ['rectangle']
Valid polygons: 0
❌ No valid polygon

[TEST] WhatsApp Image 2026-08-11 at 6.07.43 PM.jpg
Shapes in JSON: 1
Shape types: ['rectangle']
Valid polygons: 0
❌ No valid polygon

PATCH RESULT
Patched : 0
Failed  : 3

Failures:
WhatsApp Image 2026-08-11 at 6.07.44 PM (1).jpg → No valid polygon
WhatsApp Image 2026-08-11 at 6.07.44 PM.jpg → No valid polygon
WhatsApp Image 2026-08-11 at 6.07.43 PM.jpg → No valid polygon


In [60]:
# ============================================================
# FINAL YOLO DATASET COUNT
# ============================================================

for split in ["train", "val", "test"]:

    images = list(
        (yolo_dataset / "images" / split).glob("*")
    )

    labels = list(
        (yolo_dataset / "labels" / split).glob("*.txt")
    )

    print(
        f"{split.upper():5} | "
        f"images = {len(images)} | "
        f"labels = {len(labels)}"
    )

total_images = sum(
    len(list((yolo_dataset / "images" / s).glob("*")))
    for s in ["train", "val", "test"]
)

print("\nTOTAL YOLO IMAGES:", total_images)

TRAIN | images = 1909 | labels = 1909
VAL   | images = 403 | labels = 403
TEST  | images = 414 | labels = 414

TOTAL YOLO IMAGES: 2726


In [59]:
# ============================================================
# REGENERATE + WRITE ONLY THE 3 LEGACY ARDUINO IMAGES
# ============================================================

from pathlib import Path
import json
import cv2
import numpy as np
import torch
import re
import shutil
from ultralytics import SAM

project_path = Path(r"G:\AIIC")
clean_photos_path = project_path / "clean_photos"

segmentation_output = (
    project_path
    / "yolo_segmentation"
    / "segmentation_annotations"
)

yolo_dataset = (
    project_path
    / "yolo_segmentation"
    / "dataset"
)

# Exact 3 images
targets = [
    {
        "split": "train",
        "image": clean_photos_path
        / "E001_Arduino Uno"
        / "WhatsApp Image 2026-08-11 at 6.07.44 PM (1).jpg"
    },
    {
        "split": "train",
        "image": clean_photos_path
        / "E001_Arduino Uno"
        / "WhatsApp Image 2026-08-11 at 6.07.44 PM.jpg"
    },
    {
        "split": "test",
        "image": clean_photos_path
        / "E001_Arduino Uno"
        / "WhatsApp Image 2026-08-11 at 6.07.43 PM.jpg"
    },
]

device = 0 if torch.cuda.is_available() else "cpu"
sam = SAM("sam2.1_b.pt")


def mask_to_polygon(mask, w, h):

    if mask.shape != (h, w):
        mask = cv2.resize(
            mask,
            (w, h),
            interpolation=cv2.INTER_NEAREST
        )

    binary = ((mask > 0.5).astype(np.uint8) * 255)

    contours, _ = cv2.findContours(
        binary,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    if not contours:
        return None

    contour = max(contours, key=cv2.contourArea)

    if cv2.contourArea(contour) < 10:
        return None

    epsilon = 0.002 * cv2.arcLength(contour, True)

    approx = cv2.approxPolyDP(
        contour,
        epsilon,
        True
    )

    polygon = (
        approx.reshape(-1, 2)
        .astype(float)
        .tolist()
    )

    return polygon if len(polygon) >= 3 else None


patched = 0

for item in targets:

    split = item["split"]
    image_path = item["image"]

    print("\n" + "=" * 70)
    print(image_path.name)
    print("=" * 70)

    bbox_json = image_path.with_suffix(".json")

    # --------------------------------
    # LOAD BBOX
    # --------------------------------

    with open(bbox_json, "r", encoding="utf-8") as f:
        bbox_data = json.load(f)

    boxes = []
    labels = []

    for shape in bbox_data.get("shapes", []):

        if shape.get("shape_type") != "rectangle":
            continue

        pts = np.array(shape["points"], dtype=float)

        boxes.append([
            float(pts[:, 0].min()),
            float(pts[:, 1].min()),
            float(pts[:, 0].max()),
            float(pts[:, 1].max())
        ])

        labels.append(shape["label"])

    print("BBox:", len(boxes))

    image_bgr = cv2.imread(str(image_path))

    if image_bgr is None:
        print("❌ Cannot read image")
        continue

    h, w = image_bgr.shape[:2]

    shapes = []

    # --------------------------------
    # ONE BBOX AT A TIME
    # more reliable for crowded images
    # --------------------------------

    for box, raw_label in zip(boxes, labels):

        result = sam.predict(
            source=str(image_path),
            bboxes=[box],
            device=device,
            verbose=False
        )[0]

        if result.masks is None:
            continue

        masks = result.masks.data.cpu().numpy()

        if len(masks) == 0:
            continue

        polygon = mask_to_polygon(
            masks[0],
            w,
            h
        )

        if polygon is None:
            continue

        shapes.append({
            "label": raw_label,
            "points": polygon,
            "group_id": None,
            "description": "",
            "shape_type": "polygon",
            "flags": {}
        })

    print("Polygons generated:", len(shapes))

    if not shapes:
        print("❌ No usable polygons")
        continue

    # --------------------------------
    # OVERWRITE LEGACY SEG JSON
    # --------------------------------

    seg_json = (
        segmentation_output
        / "E001_Arduino Uno"
        / f"{image_path.stem}.json"
    )

    seg_json.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    with open(seg_json, "w", encoding="utf-8") as f:
        json.dump({
            "version": "5.0.1",
            "flags": {},
            "shapes": shapes,
            "imagePath": image_path.name,
            "imageData": None,
            "imageHeight": h,
            "imageWidth": w
        }, f, indent=2, ensure_ascii=False)

    # --------------------------------
    # CONVERT DIRECTLY TO YOLO
    # --------------------------------

    yolo_lines = []

    for shape in shapes:

        canonical = get_canonical_class(
            shape["label"]
        )

        if canonical is None:
            print(
                "⚠ Unknown label:",
                shape["label"]
            )
            continue

        class_id = class_to_id[canonical]

        coords = []

        for x, y in shape["points"]:

            coords.extend([
                max(0, min(1, x / w)),
                max(0, min(1, y / h))
            ])

        yolo_lines.append(
            str(class_id)
            + " "
            + " ".join(
                f"{v:.6f}"
                for v in coords
            )
        )

    if not yolo_lines:
        print("❌ No YOLO lines")
        continue

    safe_class = re.sub(
        r"[^A-Za-z0-9]+",
        "_",
        image_path.parent.name
    ).strip("_")

    output_stem = (
        f"{safe_class}__{image_path.stem}"
    )

    output_image = (
        yolo_dataset
        / "images"
        / split
        / f"{output_stem}{image_path.suffix.lower()}"
    )

    output_label = (
        yolo_dataset
        / "labels"
        / split
        / f"{output_stem}.txt"
    )

    shutil.copy2(
        image_path,
        output_image
    )

    with open(output_label, "w", encoding="utf-8") as f:
        f.write("\n".join(yolo_lines))

    patched += 1
    print("✅ WRITTEN TO", split.upper())


print("\n" + "=" * 70)
print("PATCHED:", patched, "/ 3")
print("=" * 70)


WhatsApp Image 2026-08-11 at 6.07.44 PM (1).jpg
BBox: 60
Polygons generated: 60
✅ WRITTEN TO TRAIN

WhatsApp Image 2026-08-11 at 6.07.44 PM.jpg
BBox: 66
Polygons generated: 66
✅ WRITTEN TO TRAIN

WhatsApp Image 2026-08-11 at 6.07.43 PM.jpg
BBox: 26
Polygons generated: 26
✅ WRITTEN TO TEST

PATCHED: 3 / 3


In [61]:
# ============================================================
# TRAIN YOLO INSTANCE SEGMENTATION
# ============================================================

from ultralytics import YOLO
from pathlib import Path
import torch

project_path = Path(r"G:\AIIC")

data_yaml = (
    project_path
    / "yolo_segmentation"
    / "dataset"
    / "data.yaml"
)

runs_path = (
    project_path
    / "yolo_segmentation"
    / "runs"
)

print("=" * 70)
print("YOLO INSTANCE SEGMENTATION TRAINING")
print("=" * 70)

print("Dataset :", data_yaml)
print("CUDA    :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))

# Load pretrained segmentation model
model = YOLO("yolo11n-seg.pt")

results = model.train(
    data=str(data_yaml),

    epochs=100,
    imgsz=640,
    batch=8,

    device=0,
    workers=4,

    patience=20,

    pretrained=True,
    optimizer="auto",

    amp=True,
    cache=False,

    project=str(runs_path),
    name="yolo11n_seg_aiic",
    exist_ok=True,

    plots=True,
    verbose=True
)

print("\n" + "=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)

YOLO INSTANCE SEGMENTATION TRAINING
Dataset : G:\AIIC\yolo_segmentation\dataset\data.yaml
CUDA    : True
GPU     : NVIDIA GeForce RTX 5060
New https://pypi.org/project/ultralytics/8.4.142 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.137  Python-3.14.3 torch-2.13.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5060, 8123MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=G:\AIIC\yolo_segmentation\dataset\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, 